# 04. KAN4Rec и контроль без KAN
KAN4Rec — это BERT4Rec (cloze-обучение, CE по всему каталогу), где выходная проекция multi-head attention заменена на `FastKAN([d, d])`.
Классы моделей берутся из `TianyuanYang/KAN4Rec` без изменений, данные, цикл обучения и оценка свои.
Контроль `linear`: та же модель, у которой `output_linear` заменён на `nn.Linear(d, d)`.

Ограничения: контроль не отделяет эффект LayerNorm и SiLU-ветки внутри `FastKAN` от эффекта RBF-базиса. Замена одного слоя 256×256
затрагивает менее 1% параметров модели (около 33 млн), поэтому разницу нужно интерпретировать только вместе с ДИ из ноутбука 05.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import gc
import importlib
import json
import os
import sys
import time
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn.functional as F

try:
    default_work_dir = '/content/drive/MyDrive/code2repo/kan_rec'
except ImportError:
    default_work_dir = os.path.abspath('./kan_rec')

WORK_DIR = os.environ.get('KAN_REC_DIR', default_work_dir)
THIRD_PARTY_DIR = os.path.join(os.path.dirname(WORK_DIR), 'third_party')
KAN4REC_DIR = os.path.join(THIRD_PARTY_DIR, 'KAN4Rec')
DATASET = 'ml1m'  # 'amazon' | 'ml1m' (для ml1m сначала 01b)

# одноимённые модули из других репозиториев (utils, models, config) не должны перекрыть модули KAN4Rec
for name in [n for n in sys.modules if n in ('utils', 'models', 'config') or n.startswith('models.')]:
    del sys.modules[name]

sys.path.insert(0, KAN4REC_DIR)
sys.path.insert(0, WORK_DIR)
import common
importlib.reload(common)

common.clone_repo('https://github.com/TianyuanYang/KAN4Rec', KAN4REC_DIR)
from models.kan4rec import KAN4RecModel

paths = common.dataset_paths(WORK_DIR, DATASET)
results_dir = paths['results']
os.makedirs(results_dir, exist_ok=True)
device = common.check_gpu()

KAN4Rec: 5ec42a3
Tesla T4, 14.6 GiB, sm_75, torch 2.11.0+cu130


## Данные
Айтемы 1..N, padding 0, токен `[MASK]` = N + 1. Для val на вход подаётся train + `[MASK]`, для test train + val + `[MASK]`.
`MAX_LEN`: Amazon 50 (p99 длины истории около 71, в оригинале 100), ML-1M 200.

In [3]:
data = common.load_data(paths['data'])
n_users, n_items = data['n_users'], data['n_items']
MASK_TOKEN = n_items + 1
MAX_LEN = dict(amazon=50, ml1m=200)[DATASET]
print(f'{DATASET}: users={n_users:,} items={n_items:,} MAX_LEN={MAX_LEN}')


def left_pad(items, length=MAX_LEN):
    row = np.zeros(length, np.int64)
    items = items[-length:]
    row[length - len(items):] = items
    return row


train_seqs = np.zeros((n_users, MAX_LEN), np.int64)
val_seqs = np.zeros((n_users, MAX_LEN), np.int64)
test_seqs = np.zeros((n_users, MAX_LEN), np.int64)
for user, items in enumerate(data['train']):
    train_seqs[user] = left_pad(items)
    val_seqs[user] = left_pad(items + [MASK_TOKEN])
    test_seqs[user] = left_pad(items + [int(data['val'][user]), MASK_TOKEN])
train_seqs_gpu = torch.from_numpy(train_seqs).to(device)


def make_score_fn(model):
    def score_fn(batch, split):
        seqs = val_seqs if split == 'val' else test_seqs
        inputs = torch.from_numpy(seqs[batch]).to(device)
        return model.out(model.kan4rec(inputs)[:, -1, :])
    return score_fn

ml1m: users=6,040 items=3,416 MAX_LEN=200


## Конфигурация
Параметры модели из шаблона `train_kan4rec` репозитория, кроме `max_len`. Отличие от оригинала в маскировании:
у пользователей Amazon в среднем около 9 айтемов, при `mask_prob=0.15` у многих последовательностей не оказывается масок,
поэтому `MASK_PROB=0.2` и `ENSURE_ONE=True` (гарантируется хотя бы одна маска). Схема замены 80/10/10 сохранена.

`USE_AMP` включает bf16-автокаст в обучении (оценка всегда в fp32). Для сопоставимости с fp32-запусками держите `False`,
а если включаете, то для всех seed и обоих вариантов.

In [4]:
SEEDS = [0, 1, 2, 3, 4]
VARIANTS = ['kan', 'linear']
MODEL_CFG = dict(hidden_units=256, num_blocks=2, num_heads=4, dropout=0.1)

LR = 1e-3
BATCH_SIZE = 256
MASK_PROB = 0.2
ENSURE_ONE = True
MAX_EPOCHS = dict(amazon=200, ml1m=300)[DATASET]  # одинаково для kan и linear
EVAL_EVERY = 5
PATIENCE = 4
VAL_USERS = 30000
USE_AMP = False
ALLOW_NO_RANKS = True  # готовые запуски без рангов не перезапускаются, но не попадут в бутстреп
CHECKPOINT_EVERY = 3


def new_model(seed):
    args = SimpleNamespace(model_init_seed=seed, max_len=MAX_LEN, num_items=n_items, **MODEL_CFG)
    return KAN4RecModel(args)


def replace_with_linear(model):
    size = MODEL_CFG['hidden_units']
    for block in model.kan4rec.transformer_blocks:
        block.attention.output_linear = torch.nn.Linear(size, size)


def apply_repo_init(model):
    """Применяет собственную схему инициализации репозитория, если она есть. Возвращает True, если вызвана."""
    for obj in (model, model.kan4rec):
        init_weights = getattr(obj, 'init_weights', None)
        if callable(init_weights):
            init_weights()
            return True
    return False


def apply_cloze_mask(seqs):
    valid = seqs > 0
    selected = (torch.rand(seqs.shape, device=seqs.device) < MASK_PROB) & valid

    if ENSURE_ONE:
        without_mask = ~selected.any(1)
        if without_mask.any():
            pick = (torch.rand(seqs.shape, device=seqs.device) * valid).argmax(1)
            rows = without_mask.nonzero().squeeze(1)
            selected[rows, pick[rows]] = True

    draw = torch.rand(seqs.shape, device=seqs.device)
    tokens = seqs.clone()
    tokens[selected & (draw < 0.8)] = MASK_TOKEN
    randomized = selected & (draw >= 0.8) & (draw < 0.9)
    tokens[randomized] = torch.randint(1, n_items + 1, (int(randomized.sum()),), device=seqs.device)

    labels = torch.where(selected, seqs, torch.zeros_like(seqs))
    return tokens, labels

## Инициализация контроля
Определяет, меняет ли `init_weights` репозитория веса заменённого слоя (`REINIT`). От этого зависят имена запусков контроля:
`REINIT=True` даёт `bert4rec_linear_init_seed*` (модель `BERT4Rec-linear-init`), `REINIT=False` — прежние `bert4rec_linear_seed*`.

In [5]:
torch.manual_seed(0)
control = new_model(0)
replace_with_linear(control)
layer = control.kan4rec.transformer_blocks[0].attention.output_linear
weights_before = layer.weight.detach().clone()

repo_init_found = apply_repo_init(control)
weights_after = layer.weight.detach()
REINIT = bool(repo_init_found and not torch.equal(weights_before, weights_after))

print(f'output_linear std: до init_weights {weights_before.std().item():.4f}, после {weights_after.std().item():.4f}')
print(f'init_weights найден: {repo_init_found}, меняет веса заменённого слоя: {REINIT}')
del control
gc.collect()

output_linear std: до init_weights 0.0362, после 0.0362
init_weights найден: True, меняет веса заменённого слоя: False


60

## Обучение
Прерванный запуск продолжается с последнего чекпоинта (каждая 3-я эпоха), после успешного завершения чекпоинт удаляется.

In [6]:
def run_info(variant, seed):
    if variant == 'kan':
        return f'bert4rec_kan_seed{seed}', 'BERT4Rec-kan'
    if REINIT:
        return f'bert4rec_linear_init_seed{seed}', 'BERT4Rec-linear-init'
    return f'bert4rec_linear_seed{seed}', 'BERT4Rec-linear'


def build_model(variant, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = new_model(seed)
    if variant == 'linear':
        replace_with_linear(model)
        if REINIT:
            apply_repo_init(model)
    return model.to(device)


def same_setup(stem):
    saved = common.load_run(results_dir, stem)
    return (saved.get('cfg') == MODEL_CFG and saved.get('max_len') == MAX_LEN
            and saved.get('mask_prob') == MASK_PROB and saved.get('lr') == LR
            and saved.get('batch') == BATCH_SIZE and bool(saved.get('amp', False)) == USE_AMP)


def train_epoch(model, optimizer, rng):
    model.train()
    order = torch.from_numpy(rng.permutation(n_users)).to(device)
    total_loss, n_batches = 0.0, 0

    for start in range(0, n_users, BATCH_SIZE):
        tokens, labels = apply_cloze_mask(train_seqs_gpu[order[start:start + BATCH_SIZE]])
        masked = labels > 0
        with torch.autocast(device, dtype=torch.bfloat16, enabled=USE_AMP):
            logits = model.out(model.kan4rec(tokens)[masked])  # голова считается только на маскированных позициях
        loss = F.cross_entropy(logits.float(), labels[masked])
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


def train_run(variant, seed):
    stem, model_name = run_info(variant, seed)
    model = build_model(variant, seed)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'{stem}: параметров {n_params:,}')
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    checkpoint_path = f'{results_dir}/{stem}_checkpoint.pt'
    log_path = f'{results_dir}/{stem}_log.json'
    start_epoch, best_ndcg, best_epoch, bad_evals = 1, -1, 0, 0
    best_state, log, epoch_times = None, [], []

    if os.path.exists(checkpoint_path):
        try:
            ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
            model.load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            start_epoch = ckpt['epoch'] + 1
            best_ndcg, best_epoch, bad_evals = ckpt['best'], ckpt['best_ep'], ckpt['bad']
            best_state, log = ckpt['best_state'], ckpt['log']
            epoch_times = ckpt.get('ep_times', [])
            print(f'продолжаю {stem} с эпохи {start_epoch}')
        except Exception as error:
            print(f'не удалось загрузить чекпоинт {stem}: {error}. Начинаю заново')

    score_fn = make_score_fn(model)
    rng = np.random.RandomState(seed)
    val_users = rng.choice(n_users, size=min(VAL_USERS, n_users), replace=False)
    stopped = 'max_epochs'
    epoch = start_epoch - 1

    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        started = time.time()
        loss = train_epoch(model, optimizer, rng)
        if device == 'cuda':
            torch.cuda.synchronize()
        epoch_times.append(time.time() - started)
        message = f'epoch {epoch:3d}  loss {loss:.4f}  ({epoch_times[-1]:.0f}s)'

        if epoch % EVAL_EVERY == 0:
            model.eval()
            val = common.evaluate(score_fn, data, 'val', batch_size=256, device=device, users=val_users)
            message += f"  | val R@10 {val['Recall@10']:.4f}  N@10 {val['NDCG@10']:.4f}  R@20 {val['Recall@20']:.4f}"
            log.append(dict(epoch=epoch, loss=loss, **val))
            with open(log_path, 'w') as f:
                json.dump(log, f, indent=1)

            if val['NDCG@10'] > best_ndcg:
                best_ndcg, best_epoch, bad_evals = val['NDCG@10'], epoch, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad_evals += 1
        print(message)

        if epoch % CHECKPOINT_EVERY == 0:
            torch.save(dict(
                epoch=epoch, model_state_dict=model.state_dict(), optimizer_state_dict=optimizer.state_dict(),
                best=best_ndcg, best_ep=best_epoch, bad=bad_evals, best_state=best_state,
                log=log, ep_times=epoch_times,
            ), checkpoint_path)

        if bad_evals >= PATIENCE:
            print('early stop')
            stopped = 'early_stop'
            break

    model.load_state_dict(best_state)
    model.eval()
    val_metrics, test_metrics, val_ranks, test_ranks = common.full_evaluation(score_fn, data, device)
    print('best epoch:', best_epoch, '| stopped:', stopped)
    print('VAL ', {k: round(v, 4) for k, v in val_metrics.items()})
    print('TEST', {k: round(v, 4) for k, v in test_metrics.items()})

    output = dict(
        model=model_name, variant=variant, seed=seed, cfg=MODEL_CFG, lr=LR, batch=BATCH_SIZE,
        mask_prob=MASK_PROB, ensure_one=ENSURE_ONE, max_len=MAX_LEN, max_epochs=MAX_EPOCHS,
        patience=PATIENCE, eval_every=EVAL_EVERY, best_epoch=best_epoch, epochs_run=epoch,
        stopped=stopped, n_params=n_params, sec_per_epoch=float(np.mean(epoch_times)),
        gpu=common.gpu_name(), amp=USE_AMP, repo_init_applied=bool(variant == 'linear' and REINIT),
        val=val_metrics, test=test_metrics, log=log,
    )
    common.save_run(results_dir, stem, output, val_ranks, test_ranks)

    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)
    del model, optimizer, best_state
    gc.collect()
    torch.cuda.empty_cache()

## Запуск
Для каждого seed сначала `kan`, затем `linear`, чтобы при обрыве пары оставались парными.
Время эпохи замерьте на первой эпохе: на T4 в Colab это около 156 с (Amazon).
Для долгого счёта без браузера: `jupyter nbconvert --to notebook --execute --inplace 04_kan4rec.ipynb`.

In [7]:
for seed in SEEDS:
    for variant in VARIANTS:
        stem, _ = run_info(variant, seed)
        if os.path.exists(f'{results_dir}/{stem}.json') and not same_setup(stem):
            print(f'{stem}: другие гиперпараметры в файле, переношу в _stale')
            common.move_to_stale(results_dir, stem)
        if common.is_finished(results_dir, stem, need_ranks=not ALLOW_NO_RANKS):
            print(f'[пропуск] {stem}')
            continue
        print(f'\n=== {stem} ===')
        train_run(variant, seed)
print('\nготово')


=== bert4rec_kan_seed0 ===
bert4rec_kan_seed0: параметров 4,433,513


KeyboardInterrupt: 